In [ ]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict, Any, Optional
import numpy as np
from matplotlib.ticker import ScalarFormatter
import logging
import cobra

## LOSS 

In [ ]:
# 1) Discover run folders
def find_run_dirs(base):
    candidates = []
    for entry in os.listdir(base):
        full = os.path.join(base, entry)
        if os.path.isdir(full):
            has_loss = os.path.exists(os.path.join(full, LOSS_FILE))
            has_grad = os.path.exists(os.path.join(full, GRAD_FILE))
            if has_loss and has_grad:
                candidates.append(full)
    # Also allow the base folder itself if files are here (for testing)
    if os.path.exists(os.path.join(base, LOSS_FILE)) and os.path.exists(os.path.join(base, GRAD_FILE)):
        candidates.append(base)
    # Sort with natural numeric order if names like run1, run2...
    def sort_key(p):
        name = os.path.basename(p.rstrip("/"))
        m = re.search(r"(\d+)", name)
        return (name if m is None else (name[:m.start()], int(m.group(1))))
    candidates = sorted(set(candidates), key=sort_key)
    return candidates

In [ ]:
def rename_grad_columns(df, labels):
    renamed = df.copy()
    dir_cols = [c for c in df.columns if c.startswith("dir_")]
    for i, col in enumerate(dir_cols):
        if i < len(labels):
            renamed = renamed.rename(columns={col: labels[i]})
    return renamed


In [ ]:
mode = "stat" # non producer'AWZ11expo', 'AWZ11stat'  or producer 'early' 'stat'

In [ ]:
BASE = r"Publication_Results\C13\{}\inverse_log".format(mode)
LOSS_FILE = "loss_history_inverse_log.csv"
GRAD_FILE = "grad_per_dimension_inverse_log.csv"

run_dirs = find_run_dirs(BASE)

# Prepare output directory for plots
out_dir = os.path.join(BASE, "plots_inverse_log")
os.makedirs(out_dir, exist_ok=True)


In [ ]:
custom_labels = [
    # add 'mAb' to second if producer
    "BIOMASS", "mAb" ,"GLC", "LAC", "ASN", "GLN"
]

In [ ]:
# Set global font sizes
plt.rcParams.update({
    "axes.labelsize": 16,   # x/y label size
    "axes.titlesize": 18,   # title size
    "xtick.labelsize": 14,  # x-tick label size
    "ytick.labelsize": 14,  # y-tick label size
    "legend.fontsize": 14   # legend font size
})

In [ ]:
# 2) Plot overall loss (one figure, multiple runs)
plt.figure()
legend_labels = []

for i, run in enumerate(run_dirs, start=1):
    try:
        loss_df = pd.read_csv(os.path.join(run, LOSS_FILE))
        if "Epoch" not in loss_df.columns or "Loss" not in loss_df.columns:
            continue
        plt.plot(loss_df["Epoch"], loss_df["Loss"])
        legend_labels.append(f"Run {i}")  # Custom label
    except Exception as e:
        print(f"Failed to read loss from {run}: {e}")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss per Initialization ({})".format(mode))
if legend_labels:
    plt.legend(legend_labels)
plt.tight_layout()
loss_plot_path = os.path.join(out_dir, "loss_overview_inverse_log_{}.pdf".format(mode))
plt.savefig(loss_plot_path, dpi=300, bbox_inches="tight")
plt.show()

# 3) Plot gradient per component per run (one figure per run)
saved_grad_paths = []
for i, run in enumerate(run_dirs, start=1):
    try:
        grad_df = pd.read_csv(os.path.join(run, GRAD_FILE))
        if "Epoch" not in grad_df.columns:
            continue
        grad_named = rename_grad_columns(grad_df, custom_labels)
        value_cols = [c for c in grad_named.columns if c != "Epoch"]

        fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)  # prevents clipping
        for col in value_cols:
            ax.plot(grad_named["Epoch"], grad_named[col], label=col)

        ax.set_xlabel("Epoch")
        ax.set_ylabel("Gradient magnitude")
        ax.set_title(f"Gradient per Component ({mode}) — Run {i}", wrap=True)

        # Move legend outside to free space
        ax.legend(
            ncol=1,
            loc="center left",
            bbox_to_anchor=(1.02, 0.5),
            frameon=False,
            labelspacing=0.5,
            borderaxespad=0.
        )


        out_path = os.path.join(out_dir, f"gradients_{os.path.basename(run) or 'base'}.pdf")
        fig.savefig(out_path, dpi=300, bbox_inches="tight", pad_inches=0.25)  # extra safety
        plt.close(fig)
        saved_grad_paths.append(out_path)
    except Exception as e:
        print(f"Failed to read gradients from {run}: {e}")

print("Saved plots:")
print(f"- Overall loss: {loss_plot_path}")
for p in saved_grad_paths:
    print(f"- Gradients: {p}")


## COEFFICIENTS

In [ ]:
def load_flux_summary(filepath: str) -> Tuple[List[float], List[float], List[float]]:
    experimental_fluxes: List[float] = []
    predicted_fluxes: List[float] = []
    coefficients: List[float] = []

    with open(filepath, "r") as f:
        lines = f.readlines()
        for i, line in enumerate(lines):
            if line.startswith("Experimental fluxes:"):
                experimental_fluxes = list(map(float, lines[i + 1].split()))
            elif line.startswith("Predicted fluxes:"):
                predicted_fluxes = list(map(float, lines[i + 1].split()))
            elif line.startswith("Optimized coefficients:"):
                coefficients = list(map(float, lines[i + 1].split()))
    
    return experimental_fluxes, predicted_fluxes, coefficients


In [ ]:
def load_model(model_path):
    """Load the metabolic model from a JSON file."""
    try:
        model = cobra.io.load_json_model(model_path)
        logging.info(f"Model loaded successfully from {model_path}")
        return model
    except Exception as e:
        logging.error(f"Error loading model: {e}")
        raise

In [ ]:
def load_objfind_cases_C13_nonprod(filepath: str) -> List[Dict[str, Any]]:
    df = pd.read_csv(filepath)
    cases = []

    # Define label-to-reaction ID mapping and load model
    label_to_rxn_id = {
        "BIOMASS": "F142",
        "GLC": "F105",
        "LAC": "F107",
        "ASN": "F112",
        "GLN": "F114"
    }
    model = load_model('inputs/CHOmpact_cobrapy_original.json')
    flux_to_index = {rxn.id: i for i, rxn in enumerate(model.reactions)}
    label_to_index = {label: flux_to_index[rxn_id] for label, rxn_id in label_to_rxn_id.items()}
    coeff_labels = list(label_to_rxn_id.keys())

    # Automatically detect measured and predicted flux columns
    measured_cols = sorted([col for col in df.columns if col.startswith("v_measured_")],
                           key=lambda x: int(x.split("_")[-1]))
    pred_cols = sorted([col for col in df.columns if col.startswith("v_fba_")],
                       key=lambda x: int(x.split("_")[-1]))

    for _, row in df.iterrows():
        coefficients = [
            row.get(f"c_subset_{label_to_index[label]}", 0.0)
            for label in coeff_labels
        ]
        v_measured = [row[col] for col in measured_cols]
        v_pred = [row[col] for col in pred_cols]

        case = {
            "mode": row["mode"],
            "coeff_labels": coeff_labels,
            "coefficients": coefficients,
            "v_measured": v_measured,
            "v_pred": v_pred,
            "loss": row["loss"],
            "status": row["status"]
        }
        cases.append(case)

    return cases

def load_objfind_cases_C13(filepath: str) -> List[Dict[str, Any]]:
    df = pd.read_csv(filepath)
    cases = []

    # Define label-to-reaction ID mapping and load model
    label_to_rxn_id = {
        "BIOMASS": "F142",
        "ANTI": "F143",
        "GLC": "F105",
        "LAC": "F107",
        "ASN": "F112",
        "GLN": "F114"
    }
    model = load_model('inputs/CHOmpact_cobrapy_original.json')
    flux_to_index = {rxn.id: i for i, rxn in enumerate(model.reactions)}
    label_to_index = {label: flux_to_index[rxn_id] for label, rxn_id in label_to_rxn_id.items()}
    coeff_labels = list(label_to_rxn_id.keys())

    # Automatically detect measured and predicted flux columns
    measured_cols = sorted([col for col in df.columns if col.startswith("v_measured_")],
                           key=lambda x: int(x.split("_")[-1]))
    pred_cols = sorted([col for col in df.columns if col.startswith("v_fba_")],
                       key=lambda x: int(x.split("_")[-1]))

    for _, row in df.iterrows():
        coefficients = [
            row.get(f"c_subset_{label_to_index[label]}", 0.0)
            for label in coeff_labels
        ]
        v_measured = [row[col] for col in measured_cols]
        v_pred = [row[col] for col in pred_cols]

        case = {
            "mode": row["mode"],
            "coeff_labels": coeff_labels,
            "coefficients": coefficients,
            "v_measured": v_measured,
            "v_pred": v_pred,
            "loss": row["loss"],
            "status": row["status"]
        }
        cases.append(case)

    return cases


In [ ]:
AWZ11expo_0_001_celltarget = load_flux_summary(r"Publication_Results\C13\AWZ11expo\inverse_log\final\flux_summary.txt")
AWZ11stat_0_001_celltarget = load_flux_summary(r"Publication_Results\C13\AWZ11stat\inverse_log\final\flux_summary.txt")

early_0_001_celltarget = load_flux_summary(r"Publication_Results\C13\early\inverse_log\final\flux_summary.txt")
stat_0_001_celltarget  = load_flux_summary(r"Publication_Results\C13\stat\inverse_log\final\flux_summary.txt")

In [ ]:
# benchmark results
prod_filepath_0_001 = r"Publication_Results\C13\ObjFind\prod_objfind_Sv0.001.csv"
nonprod_filepath_0_001 = r"Publication_Results\C13\ObjFind\nonprod_objfind_Sv0.001.csv"

prod_cases_list_0_001_prod = load_objfind_cases_C13(prod_filepath_0_001)
nonprod_cases_list_0_001_prod = load_objfind_cases_C13_nonprod(nonprod_filepath_0_001)

In [ ]:
prod_filepath_0 = r"Publication_Results\C13\ObjFind\prod_objfind_Sv0.csv"
nonprod_filepath_0 = r"Publication_Results\C13\ObjFind\nonprod_objfind_Sv0.csv"

prod_cases_list_0_prod = load_objfind_cases_C13(prod_filepath_0)
nonprod_cases_list_0_prod = load_objfind_cases_C13_nonprod(nonprod_filepath_0)

In [ ]:
# Combine productiveness and S.v into a structured dictionary
cases_dict = {
    "0_001": {
        "early": next(c for c in prod_cases_list_0_001_prod if c["mode"] == "early"),
        "stat": next(c for c in prod_cases_list_0_001_prod if c["mode"] == "stat"),
        "AWZ11expo": next(c for c in nonprod_cases_list_0_001_prod if c["mode"] == "AWZ11expo"),
        "AWZ11stat": next(c for c in nonprod_cases_list_0_001_prod if c["mode"] == "AWZ11stat"),
    }
}

In [ ]:
output_dir = r"Publication_Results\C13\coefficients"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
def plot_coefficients_group_prod(modes: List[str], sv: str, cases_dict: Dict, celltarget_map: Dict, output_dir: str):
    import matplotlib.pyplot as plt
    import numpy as np
    import os

    coeff_labels = ["BIOMASS", "ANTI", "GLC", "LAC", "ASN", "GLN"]
    xtick_labels = ["BIOMASS", "mAb", "GLC", "LAC", "ASN", "GLN"]

    label_synonyms = {
    "mAb": "ANTI",  # convert CellTarget's mAb to ANTI
    "ANTI": "ANTI"  # ObjFind's ANTI stays as ANTI
}
    
    x = np.arange(len(coeff_labels))
    width = 0.35

    for mode in modes:
        case = cases_dict[sv][mode]
        coeffs_objfind_raw = case["coefficients"]
        coeffs_objfind_labels = case["coeff_labels"]

        coeffs_fba_labels = celltarget_map[mode][0]
        coeffs_fba_raw = celltarget_map[mode][1]
        print(coeffs_objfind_raw)
        print(coeffs_fba_raw)


        # Align both to full coeff_labels
        def align(coeffs_raw, labels):
            # map all incoming labels to canonical
            mapped_labels = [label_synonyms.get(lbl, lbl) for lbl in labels]
            out = []
            for lbl in coeff_labels:
                try:
                    idx = mapped_labels.index(lbl)
                    out.append(coeffs_raw[idx])
                except ValueError:
                    out.append(np.nan)
            return out

        coeffs_objfind_full = align(coeffs_objfind_raw, coeffs_objfind_labels)
        coeffs_fba_full = align(coeffs_fba_raw, coeffs_fba_labels)

        # === Plot
        plt.figure(figsize=(8, 4))
        
        for i in x:
            # === Determine label for this index
            label_objfind = "ObjFind"
            label_fba = "CellTarget"
        
            # === ObjFind Bar or Cross
            if not np.isnan(coeffs_objfind_full[i]):
                plt.bar(x[i] - width/2, coeffs_objfind_full[i], width, color='steelblue', label=label_objfind if i == 0 else "")
        
            # === CellTarget Bar or Cross
            if not np.isnan(coeffs_fba_full[i]):
                plt.bar(x[i] + width/2, coeffs_fba_full[i], width, color='gray', label=label_fba if i == 0 else "")



        plt.xticks(x, xtick_labels, rotation=45)
        plt.ylabel("Coefficient value")
        plt.title(f"{mode.capitalize()} – S.v <={sv.replace('_', '.')}")
        plt.legend()
        plt.grid(True, axis="y", linestyle="--", alpha=0.5)
        plt.tight_layout()
        #plt.show()

        filename = f"coef_{mode}_{sv}_png.pdf"
        filepath = os.path.join(output_dir, filename)
        plt.savefig(filepath)
        plt.close()


def plot_coefficients_group_nonprod(modes: List[str], sv: str, cases_dict: Dict, celltarget_map: Dict, output_dir: str):
    import matplotlib.pyplot as plt
    import numpy as np
    import os

    coeff_labels = ["BIOMASS", "ANTI", "GLC", "LAC", "ASN", "GLN"]
    xtick_labels = ["BIOMASS", "mAb", "GLC", "LAC", "ASN", "GLN"]
    x = np.arange(len(coeff_labels))
    width = 0.35

    for mode in modes:
        case = cases_dict[sv][mode]
        coeffs_objfind_raw = case["coefficients"]
        coeffs_objfind_labels = case["coeff_labels"]

        coeffs_fba_labels = celltarget_map[mode][0]
        coeffs_fba_raw = celltarget_map[mode][1]
        print(coeffs_objfind_raw)
        print(coeffs_fba_raw)


        # Align both to full coeff_labels
        def align(coeffs_raw, labels):
            out = []
            for lbl in coeff_labels:
                try:
                    idx = labels.index(lbl)
                    out.append(coeffs_raw[idx])
                except ValueError:
                    out.append(np.nan)
            return out

        coeffs_objfind_full = align(coeffs_objfind_raw, coeffs_objfind_labels)
        coeffs_fba_full = align(coeffs_fba_raw, coeffs_fba_labels)

        # === Plot
        plt.figure(figsize=(8, 4))
        
        for i in x:
            # === Determine label for this index
            label_objfind = "ObjFind*"
            label_fba = "CellTarget"
        
            # === ObjFind Bar or Cross
            if not np.isnan(coeffs_objfind_full[i]):
                plt.bar(x[i] - width/2, coeffs_objfind_full[i], width, color='steelblue', label=label_objfind if i == 0 else "")
            else:
                if not (mode in prod_modes and coeff_labels[i] == "ANTI" and np.isnan(coeffs_objfind_full[i])):
                    color = 'black' if mode in nonprod_modes else 'red'
                    plt.text(x[i] - width/2, 0.01, '×', ha='center', va='bottom', color=color)
        
            # === CellTarget Bar or Cross
            if not np.isnan(coeffs_fba_full[i]):
                plt.bar(x[i] + width/2, coeffs_fba_full[i], width, color='gray', label=label_fba if i == 0 else "")
            else:
                plt.text(x[i] + width/2, 0.01, '×', ha='center', va='bottom', color='black')


        plt.xticks(x, xtick_labels, rotation=45)
        plt.ylabel("Coefficient value")
        plt.title(f"{mode.capitalize()} – S.v <={sv.replace('_', '.')}")
        plt.legend()
        plt.grid(True, axis="y", linestyle="--", alpha=0.5)
        plt.tight_layout()
        #plt.show()

        filename = f"coef_{mode}_{sv}_png.pdf"
        filepath = os.path.join(output_dir, filename)
        plt.savefig(filepath)
        plt.close()

In [ ]:
prod_modes = ["early", "stat"]
nonprod_modes = ["AWZ11expo", "AWZ11stat"]

prod_labels = ["BIOMASS", "mAb", "GLC", "LAC", "ASN", "GLN"]
nonprod_labels = ["BIOMASS", "GLC", "LAC", "ASN", "GLN"]

celltarget_map = {
    "early": (prod_labels, early_0_001_celltarget[2]),
    "stat": (prod_labels, stat_0_001_celltarget[2]),
    "AWZ11expo": (nonprod_labels, AWZ11expo_0_001_celltarget[2]),
    "AWZ11stat": (nonprod_labels, AWZ11stat_0_001_celltarget[2]),
}

In [ ]:
plot_coefficients_group_prod(prod_modes, "0_001", cases_dict, celltarget_map, output_dir)
plot_coefficients_group_nonprod(nonprod_modes, "0_001", cases_dict, celltarget_map, output_dir)

## FLUXES

In [ ]:
output_dir = r"ThesisResults_V5\C13\fluxes"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
def plot_flux_fit_combined_prod(case: Dict[str, Any], summary: Tuple[List[float], List[float], List[float]], S_V: str) -> None:
    """
    Plot fluxes for production case: BIOMASS, mAb, GLC, LAC, ASN, GLN
    """
    import numpy as np
    import matplotlib.pyplot as plt

    flux_labels = ["BIOMASS", "mAb", "GLC", "LAC", "ASN", "GLN"]
    flux_indices = [142, 143, 105, 107, 112, 114]
    x = np.arange(len(flux_labels))

    measured_fluxes = np.array(summary[0])[0:6]
    predicted_celltarget = np.array(summary[1])[0:6]
    predicted_objfind = np.array(case["v_pred"])[flux_indices]

    plt.figure(figsize=(8, 4))
    for i in x:
        if not np.isnan(measured_fluxes[i]):
            plt.scatter(x[i], measured_fluxes[i], color='blue', marker='o', label="Measured" if i == 0 else "")
        if not np.isnan(predicted_objfind[i]):
            plt.scatter(x[i], predicted_objfind[i], color='red', marker='x', label="ObjFind*" if i == 0 else "")
        if not np.isnan(predicted_celltarget[i]):
            plt.scatter(x[i], predicted_celltarget[i], color='green', marker='^', label="CellTarget" if i == 0 else "")
        if not np.isnan(measured_fluxes[i]) and not np.isnan(predicted_objfind[i]):
            plt.plot([x[i], x[i]], [measured_fluxes[i], predicted_objfind[i]], color='black', linestyle=':', alpha=0.6)
        if not np.isnan(measured_fluxes[i]) and not np.isnan(predicted_celltarget[i]):
            plt.plot([x[i], x[i]], [measured_fluxes[i], predicted_celltarget[i]], color='gray', linestyle='--', alpha=0.4)

    plt.xticks(x, flux_labels, rotation=45)
    plt.ylabel("Flux (mmol/[g hr])")
    plt.title(f"{case['mode'].capitalize()} – (S.v<={S_V.replace('_', '.')})")
    plt.legend()
    plt.grid(True, axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    # plt.show()

    filename = f"fluxfit_combined_{case['mode']}_{S_V}.pdf"
    filepath = os.path.join(output_dir, filename)
    plt.savefig(filepath)
    plt.close()


In [ ]:
def plot_flux_fit_combined_nonprod(case: Dict[str, Any], summary: Tuple[List[float], List[float], List[float]], S_V: str) -> None:
    """
    Plot fluxes for non-production case: BIOMASS, GLC, LAC, ASN, GLN
    """
    import numpy as np
    import matplotlib.pyplot as plt

    flux_labels = ["BIOMASS", "GLC", "LAC", "ASN", "GLN"]
    flux_indices = [142, 105, 107, 112, 114]
    x = np.arange(len(flux_labels))

    idx_map = [0, 1, 2, 3, 4]
    measured_fluxes = np.array(summary[0])[idx_map]
    predicted_celltarget = np.array(summary[1])[idx_map]
    predicted_objfind = np.array(case["v_pred"])[flux_indices]

    plt.figure(figsize=(7, 4))
    for i in x:
        if not np.isnan(measured_fluxes[i]):
            plt.scatter(x[i], measured_fluxes[i], color='blue', marker='o', label="Measured" if i == 0 else "")
        if not np.isnan(predicted_objfind[i]):
            plt.scatter(x[i], predicted_objfind[i], color='red', marker='x', label="ObjFind*" if i == 0 else "")
        if not np.isnan(predicted_celltarget[i]):
            plt.scatter(x[i], predicted_celltarget[i], color='green', marker='^', label="CellTarget" if i == 0 else "")
        if not np.isnan(measured_fluxes[i]) and not np.isnan(predicted_objfind[i]):
            plt.plot([x[i], x[i]], [measured_fluxes[i], predicted_objfind[i]], color='black', linestyle=':', alpha=0.6)
        if not np.isnan(measured_fluxes[i]) and not np.isnan(predicted_celltarget[i]):
            plt.plot([x[i], x[i]], [measured_fluxes[i], predicted_celltarget[i]], color='gray', linestyle='--', alpha=0.4)

    plt.xticks(x, flux_labels, rotation=45)
    plt.ylabel("Flux (mmol/[g hr])")
    plt.title(f"{case['mode'].capitalize()} – (S.v<={S_V.replace('_', '.')})")
    plt.legend()
    plt.grid(True, axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    # plt.show()

    filename = f"fluxfit_combined_{case['mode']}_{S_V}.pdf"
    filepath = os.path.join(output_dir, filename)
    plt.savefig(filepath)
    plt.close()


In [ ]:
plot_flux_fit_combined_prod(case=cases_dict["0_001"]["early"],summary=early_0_001_celltarget, S_V="0_001")
plot_flux_fit_combined_prod(case=cases_dict["0_001"]["stat"],summary=stat_0_001_celltarget,S_V="0_001")
plot_flux_fit_combined_nonprod(case=cases_dict["0_001"]["AWZ11expo"],summary=AWZ11expo_0_001_celltarget,S_V="0_001")
plot_flux_fit_combined_nonprod(case=cases_dict["0_001"]["AWZ11stat"],summary=AWZ11stat_0_001_celltarget,S_V="0_001")